# (d) 入区堆积 game-over 实验(D65)

**目的**:验证 game-over on 入区堆积(set_pending>K 判负,K=6)能否把「因果」拉近门口、逼出入区放行时机,让 PPO/IQL 不再塌成 set-everything。

**对照**:留了无 go 的旧 run(`ppo_cur_gnn_s0`/`iql_gnn_s0`)做消融。看:val 完成率能否上走、塌缩犯不犯、判负率是否随训练下降(学会克制)。

**注意**:game-over 数据单独存 `offline_shards_train_go`;env fix (a) livelock 已修,旧数据作废、需重生成。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## ① 生成离线数据(game-over 版,本地已生成+拷贝的话跳过)
本地: `python scripts/33_build_offline_dataset.py train 24 6 5000 1 6` → 拷 `artifacts/dataset/offline_shards_train_go/` 到服务器。末两位 `1 6`=gameover K=6。规则基线很快判负 → 数据小而干净(专家为主)。

In [ ]:
!{PREFIX} scripts/33_build_offline_dataset.py train 24 70 5000 1 6   # 末两位: gameover=1, K=6 -> 存 offline_shards_train_go


## ② IQL(离线,game-over 口径)
数据+评估 env 都开 game-over(口径一致)。重跑=续训。⚠️ 首次评估看是否报错(IQL 加 target 网络后的新代码)。

In [ ]:
!{PREFIX} scripts/34_train_iql.py \
    --dataset artifacts/dataset/offline_shards_train_go \
    --gameover-entry-pileup --entry-pileup-K 6 \
    --encoder gnn --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --iters 3000 --batch 128 --eval-every 150 --gamma 0.99 --tau 0.7 --beta 3.0 \
    --run-tag iql_go_s0 --resume


## ③ PPO(在线,game-over + 有界奖励 + 松KL + curriculum)
乱放入区 → set_pending>6 → 立即判负、−50 贴因落地。看塌缩犯不犯、判负率降不降。重跑=续训。

In [ ]:
!{PREFIX} scripts/13_train_ppo.py --encoder gnn \
    --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --gameover-entry-pileup --entry-pileup-K 6 \
    --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 0.9 \
    --train-eval-days 6 --iters 80 --steps-per-iter 2048 --eval-every 5 \
    --early-stop-patience 6 --run-tag ppo_go_s0 --resume


## ④ 对比(go vs 无go 消融 + 参照线)

In [ ]:
import json, torch
from pathlib import Path
rep=Path('artifacts/reports')
def load(tag):
    if (rep/f'{tag}.json').exists(): return json.loads((rep/f'{tag}.json').read_text())
    if (rep/f'{tag}_last.pt').exists(): return torch.load(rep/f'{tag}_last.pt',map_location='cpu',weights_only=False)
    return None
runs=[('PPO wo','ppo_cur_gnn_s0'),('PPO go','ppo_go_s0'),('IQL wo','iql_gnn_s0'),('IQL go','iql_go_s0')]
print(f"{'method':<9}{'best_val':>10}{'best@it':>9}{'gridlock%':>11}")
loaded={}
for name,tag in runs:
    d=load(tag); loaded[name]=d
    if not d: print(f'{name:<9}{"(none)":>10}'); continue
    b=d.get('best') or {}; ev=d.get('evals') or []; go='-'
    if ev:
        last=ev[-1]; nd=last.get('eval_n_days',1) or 1; go=f"{last.get('eval_entry_gridlocks',0)/nd:.0%}"
    print(f"{name:<9}{str(b.get('completion','-')):>10}{str(b.get('iter','-')):>9}{go:>11}")
base=next((loaded[n].get('baselines') for n,_ in runs if loaded.get(n) and loaded[n].get('baselines')),{})
for k in ['signaller_replay','lookup_most_frequent','greedy_set']:
    if k in base: print(f"{k:<20}{base[k]['eval_completion_rate']:.3f}")
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    plt.figure(figsize=(9,3.2))
    for name,_ in runs:
        d=loaded.get(name)
        if d and d.get('evals'):
            ev=d['evals']; plt.plot([e['iter'] for e in ev],[e['eval_completion_rate'] for e in ev],marker='.',label=name)
    plt.axhline(1.0,ls='--',c='g',lw=.8,label='signaller~1.0'); plt.xlabel('iter'); plt.ylabel('val completion')
    plt.legend(fontsize=8); plt.title('game-over ablation'); plt.tight_layout(); plt.savefig('artifacts/reports/gameover_compare.png',dpi=110)
    print('fig -> artifacts/reports/gameover_compare.png')
except Exception as e: print('plot skip',e)


## 下一步
- 若 go 明显好过无 go:game-over 对症 → 再考虑加 (e) WAIT 锚。
- 若 go 仍上不去:加 (e);还不行回到任务结构再想。
- 判负率应随训练下降(学会克制);若一直 100% 判负=K 太紧或学不动。
- test 全程封存(D43)。